## Libraries

In [1]:
# Date and time utilities, used for timestamps, windows and logging
from datetime import datetime,UTC

# Path handling for filesystem-safe and cross-platform paths
from pathlib import Path

# Used for rate limiting, sleeps and retry backoff
import time

# Regular expressions for parsing and cleaning text fields
import re

# JSON handling for API payloads and intermediate artifacts
import json

# SQLite driver, used as the main storage backend
import sqlite3

# Operating system utilities (paths, environment variables, filesystem ops)
import os

# Core data manipulation library
import pandas as pd

# Google API client used to access YouTube Data API
from googleapiclient.discovery import build

# Explicit handling of Google API HTTP errors
from googleapiclient.errors import HttpError

## User interface implementation

In [1]:
# Load acquisition pipeline
%run Acquisition_and_storage.ipynb

# Load SQL cleaning logic
%run Data_quality_and_analysis.ipynb

# Load LLM process
%run LLM_process.ipynb


# Normalize a keyword into a compact identifier.
# This is used consistently for table names, file names and folders.
def normalize_keyword(keyword: str) -> str:
    return (keyword.lower().strip().replace(" ", "").replace("-", ""))


# Returns the directory of the current script or notebook.
# Works both in .py execution and inside Jupyter.
def get_current_working_dir():
    try:
        # Script execution
        return Path(__file__).resolve().parent
    except NameError:
        # Notebook execution
        return Path(os.getcwd()).resolve()


# Collect all runtime parameters interactively.
# Defaults are provided to keep the pipeline easy to run.
def ask_user_parameters():
    print("\nPARAMETER SETUP\n")

    topic = input("Search topic: ").strip()
    if not topic:
        raise ValueError("Topic is required")
        
    youtube_api_key_file = input("YouTube API key file [or automatically picks:youtube_keys.txt]: ").strip()or "youtube_keys.txt"

    youtube_end_date_raw = (input("YouTube end date (publishedBefore) [dd/mm/yyyy format, if empty automatically picks today]: ").strip()) or datetime.today().strftime("%d/%m/%Y")

    # Parse the end date and force it to the end of the selected day
    if youtube_end_date_raw:
        youtube_end_date = datetime.strptime(
        youtube_end_date_raw, "%d/%m/%Y").replace(hour=23, minute=59, second=59, tzinfo=UTC)
    else:
        youtube_end_date = None


    youtube_lookback_days = int(input("YouTube lookback days [automatically picks: 14]: ").strip() or 14)

    youtube_window_days = int(input("YouTube window days [automatically picks: 1]: ").strip() or 1)

    base_dir = get_current_working_dir()
    default_trends_dir = base_dir

    # Folder containing Google Trends CSV files
    trends_csv_root = input(f"Google Trends CSV folder [or automatically picks the folder where you are working]: ").strip()

    if not trends_csv_root:
        trends_csv_root = str(default_trends_dir)

    openai_key_file = input("OpenAI key file [or automatically picks:Open_AI_key.txt]: ").strip()or "Open_AI_key.txt"
    
    output_dir = (input("Output directory for analysis results [or automatically creates a output folder where you are working]: ").strip()
                  or "Outputs")

    return {"topic": topic,
            "youtube_api_key_file": youtube_api_key_file,
            "youtube_end_date": youtube_end_date,
            "youtube_lookback_days": youtube_lookback_days,
            "youtube_window_days": youtube_window_days,
            "trends_csv_root": trends_csv_root,
            "output_dir":output_dir,
            "openai_key_file":openai_key_file,}


# Run the entire pipeline end-to-end in interactive mode.
# This orchestrates acquisition, cleaning, EDA and LLM optimization.
def run_pipeline_interactive():

    params = ask_user_parameters()

    # Normalize topic once for folder and database naming
    topic = normalize_keyword(params["topic"])
    output_root = Path(f"{topic}_Outputs").resolve()
    output_root.mkdir(parents=True, exist_ok=True)


    # Standard subfolders used by the pipeline
    paths = {"storage": output_root / "storage",
             "exports": output_root / "exports",
             "analysis_complete": output_root / "analysis"/"complete",
             "analysis_interesting": output_root / "analysis"/"interesting",}

    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)


    # Print resolved configuration for traceability
    print("\n Pipeline configuration parameters")
    for k, v in params.items():
        print(f"{k:25s}: {v}")
    print(f"{'output_root':25s}: {output_root}")

    # SQLite database used as single source of truth
    sqlite_db_path = paths["storage"] / f"{topic}_storage.db"

    print("\n Starting pipeline \n")
    start_time1 = datetime.now(UTC)

    # Acquire YouTube data and ingest Google Trends
    print("\n            Searching content on Youtube and uploading Google Trends\n")
    summary = run_full_acquisition_and_storage(youtube_api_key_file=params["youtube_api_key_file"],
                                               youtube_topic=params["topic"],
                                               youtube_lookback_days=params["youtube_lookback_days"],
                                               youtube_window_days=params["youtube_window_days"],
                                               youtube_end_date=params["youtube_end_date"],
                                               trends_keyword=params["topic"],
                                               trends_csv_root=params["trends_csv_root"],
                                               sqlite_db_path=str(sqlite_db_path),
                                               output_dir=str(paths["exports"]),)

    # Clean data, compute metrics and run EDA
    print("\n            Cleaning and analizing data\n")
    
    cleaning_summary = cleaning_and_analysis(topic=topic,storage_path=str(sqlite_db_path),
                                             output_dir=str(paths["exports"]))

    print("\n            Explorative data analysis in process\n")
    run_EDA(prefix=params["topic"],sqlite_db_path=str(sqlite_db_path),
            complete_output_dir=str(paths["analysis_complete"]),
            interesting_output_dir=str(paths["analysis_interesting"]),
            export_dir=str(paths["exports"]),generate_plots=True,)

    
    # Run LLM optimization using database-backed inputs
    print("\n            Starting LLM process\n")
    start_time = datetime.now(UTC)

    results = run_llm_optimization(topic=params["topic"],storage_path=str(sqlite_db_path),
                                   export_dir=str(paths["exports"]),openai_key_file=params["openai_key_file"])

    end_time = datetime.now(UTC)

    end_time1 = datetime.now(UTC)

    print("\n All process completed ")
    
    # Collect warning conditions without printing
    warnings = ["Analysis in not accurate"]
    
    # YouTube acquisition warnings
    yt_completed = summary.get("completed", {})
    
    if not yt_completed:
        warnings.append("YouTube acquisition status unavailable")
    
    if yt_completed.get("discovery") == 0:
        warnings.append("YouTube discovery interrupted due to quota limits")
    
    if yt_completed.get("video_statistics") == 0:
        warnings.append("YouTube video statistics incomplete due to quota limits")
    
    if yt_completed.get("channel_statistics") == 0:
        warnings.append("YouTube channel statistics incomplete due to quota limits")
    
    # Google Trends temporal coverage warnings
    if summary.get("google_top_date") is False:
        warnings.append("Google Trends TOP temporal coverage mismatch — end date does not match expected reference date")

    if summary.get("google_rising_date") is False:
        warnings.append("Google Trends RISING temporal coverage mismatch — end date does not match expected reference date")

    # Google Trends cleaning warnings
    if isinstance(cleaning_summary, dict):
    
        if cleaning_summary.get("google_trends_rising_incomplete") == 1:
            warnings.append("Google Trends RISING dataset appears incomplete — check above for more explanation")
    
        if cleaning_summary.get("google_trends_top_incomplete") == 1:
            warnings.append("Google Trends TOP dataset appears incomplete — check above for more explanation")
    
    print(f"Total duration: {end_time1 - start_time1}")

    print("\n LLM CONTENT STRATEGY DECISION\n")
    strategy = results.get("strategy_decision", {})

    if strategy:
        print(f"Preferred content type : {strategy.get('preferred_content_type')}")
        print(f"Confidence level       : {strategy.get('confidence_level')}")
        print(f"Justification          : {strategy.get('justification')}\n")
    else:
        print("No strategy decision available\n")

    videos = results.get("videos", [])

    if videos:
        print("VIDEO IDEAS\n")
        for i, v in enumerate(videos, 1):
            print(f"VIDEO {i}")
            print(f"Title : {v.get('title')}")
            print(f"Tags  : {', '.join(v.get('tags', []))}")
            print(f"Keywords : {', '.join(v.get('keywords', []))}")
            print(f"Publish time : {v.get('best_publish_time')}\n")
    else:
        print("No video ideas generated\n")

    reels = results.get("reels", [])
    
    if reels:
        print("REEL IDEAS\n")
        for i, r in enumerate(reels, 1):
            print(f"REEL {i}")
            print(f"Title : {r.get('title')}")
            print(f"Tags  : {', '.join(r.get('tags', []))}")
            print(f"Keywords : {', '.join(r.get('keywords', []))}")
            print(f"Publish time : {r.get('best_publish_time')}\n")
    else:
        print("No reel ideas generated\n")

    # Print warnings only if at least one is present
    if len(warnings) > 1:
        title = " Pipeline warnings "
        # Build all lines to be displayed inside the box
        lines = [title] + warnings
        # Compute box width
        max_len = max(len(line) for line in lines)
        # Print boxed warnings
        print("\n")
        for line in lines:
            print(f"* {line.ljust(max_len)} ")

    return summary

## Run this to start the complete pipeline

In [2]:
run_pipeline_interactive()


PARAMETER SETUP



Search topic:  data science
YouTube API key file [or automatically picks:youtube_keys.txt]:  
YouTube end date (publishedBefore) [dd/mm/yyyy format, if empty automatically picks today]:  
YouTube lookback days [automatically picks: 14]:  
YouTube window days [automatically picks: 1]:  
Google Trends CSV folder [or automatically picks the folder where you are working]:  
OpenAI key file [or automatically picks:Open_AI_key.txt]:  
Output directory for analysis results [or automatically creates a output folder where you are working]:  



 Pipeline configuration parameters
topic                    : data science
youtube_api_key_file     : youtube_keys.txt
youtube_end_date         : 2026-02-06 23:59:59+00:00
youtube_lookback_days    : 14
youtube_window_days      : 1
trends_csv_root          : /Users/nicolo/Desktop/Data_managment/Progetto man/Trends/Progetto
output_dir               : Outputs
openai_key_file          : Open_AI_key.txt
output_root              : /Users/nicolo/Desktop/Data_managment/Progetto man/Trends/Progetto/datascience_Outputs

 Starting pipeline 


            Searching content on Youtube and uploading Google Trends

 Loaded 15 YouTube API keys
Topic         : data science
Lookback days : 14
Start date    : 2026-01-23
End date      : 2026-02-06
Window days   : 1

 Video discovery
Keyword variants: data science, datascience, data_science, data-science, data - science

API KEY 1/15
  Variant 'data science'
    Window 1 of 14 from 2026-01-23 to 2026-01-24
    Window 2 of 14 from 2026-01-24 to 2026-01-25


{'counts': {'youtube_videos': 3261,
  'youtube_stats': 3260,
  'youtube_channels': 1794,
  'google_trends_top': 59,
  'google_trends_rising': 159},
 'completed': {'discovery': 1, 'video_statistics': 1, 'channel_statistics': 1},
 'google_top_date': True,
 'google_rising_date': True}